<a href="https://colab.research.google.com/github/tom-howes/bone-fracture-classifier/blob/main/bone_fracture_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 19.6 MB/s eta 0:00:00


### Data

In [49]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset

class FractureDataset(Dataset):
    def __init__(self, img_dir=None, label_dir=None, transform=None, preloaded_data=None, preloaded_labels=None):
        self.transform = transform
        self.num_classes = 7

        self.class_names = ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus fracture', 'humerus', 'shoulder fracture', 'wrist positive']
        if preloaded_data is not None:
            # Preloaded tensor mode
            self.images = preloaded_data
            self.labels = preloaded_labels
            self.preloaded = True
        else:
            # File-based mode
            self.img_dir = img_dir
            self.label_dir = label_dir
            self.images = os.listdir(img_dir)
            self.preloaded = False

    def __len__(self):
        return(len(self.images))

    def __getitem__(self, idx):
        # preloaded tensor
        if self.preloaded:
            return self.images[idx], self.labels[idx]

        # File based loading

        img_name = self.images[idx]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")

        # Multi-hot encoded label [0, 0, 1, 0...]
        label = torch.zeros(self.num_classes)

        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(self.label_dir, label_name)

        has_label = False
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    if line.strip():
                        class_id = int(line.split()[0])
                        label[class_id] = 1

        if self.transform:
            img = self.transform(img)

        return img, label


### Model

In [51]:
import torch.nn as nn

class FractureCNN(nn.Module):

    def __init__(self, num_classes=7):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(256 * 28 * 28, 256),
          nn.ReLU(),
          nn.Dropout(0.5),
          nn.Linear(256, 7),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [6]:
!cp -r /content/drive/MyDrive/datasets/test /content/test

## Simple CNN
Conv3-32
MaxPool

Conv3-64
MaxPool

Conv3-128
MaxPool

FC-256
Dropout=0.5
BCELoss

In [58]:
from torchmetrics import MetricCollection
from torchmetrics import F1Score, Precision, Recall, Accuracy
from torchvision import transforms
from torch.utils.data import DataLoader
import time

BATCH_SIZE = 32
# Transforms for train and validation datasets
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])


test_dataset = FractureDataset('/content/drive/MyDrive/datasets/test/images', '/content/test/labels')

# Load everything into memory
data = torch.load('/content/drive/MyDrive/preprocessed_data.pt')


train_dataset = FractureDataset(preloaded_data=data['train_data'], preloaded_labels=data['train_labels'], transform=train_transform)
val_dataset = FractureDataset(preloaded_data=data['val_data'], preloaded_labels=data['val_labels'], transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

def train(epochs, cnn, lr=1e-5, weight_decay=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = cnn
    model = model.to(device)
    all_labels = torch.cat([labels[:, :7] for _, labels in train_loader])
    pos_counts = all_labels.sum(dim=0)
    neg_counts = len(all_labels) - pos_counts
    pos_weight = neg_counts / (pos_counts + 1e-6)

    print(pos_weight)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Validation metrics
    metrics = MetricCollection({
        'accuracy' : Accuracy(task='multilabel', num_labels=7),
        'f1' : F1Score(task='multilabel', num_labels=7),
        'precision' : Precision(task='multilabel', num_labels=7),
        'recall' : Recall(task='multilabel', num_labels=7),
    })

    metrics = metrics.to(device)

    for epoch in range(epochs):
        ### Training phase
        model.train() # Set to train mode
        running_loss = 0
        num_batches = 0
        for inputs, labels in train_loader:
            # Move data to device
            inputs = inputs.to(device)
            labels = labels.to(device)
            labels = labels[:, :7]
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            num_batches += 1

            if num_batches % 50 == 0:
                running_loss_avg = running_loss / (num_batches * BATCH_SIZE)
                print(f"Running Loss: {running_loss_avg:.4f}")


        train_loss = running_loss / len(train_loader) # final train avg

        ### Validation phase
        model.eval()
        metrics.reset() # Reset at start of epoch
        with torch.no_grad():

            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                labels = labels[:, :7]
                outputs = model(inputs)
                predictions = (torch.sigmoid(outputs) > 0.5).int()
                print(predictions.sum(dim=0))

                # Update metrics
                metrics.update(predictions, labels)

        # Compute final metrics
        final_metrics = metrics.compute()
        # Print epoch summary
        print(f"Epoch {epoch + 1}/{epochs}:\tTrain Loss: {train_loss:.3f}\n___Validation___\n Acc: {final_metrics['accuracy']:.3f} F1: {final_metrics['f1']:.3f} Precision: {final_metrics['precision']:.3f} Recall: {final_metrics['recall']:.3f}")

# train(50)

## Deeper CNN
Conv3-32
MaxPool

Conv3-64
MaxPool

Conv3-128
MaxPool

Conv3-256
MaxPool

Conv3-512
MaxPool

FC-256
BCELoss

In [34]:
train(50, FractureCNN())

Using device: cuda
Running Loss: 0.3252
Running Loss: 0.3192
Epoch 1/50:	Train Loss: 10.1537
___Validation___
 Acc: 0.876171886920929 F1: 0.1120448186993599 Precision: 0.5405405163764954 Recall: 0.0625
Running Loss: 0.2980
Running Loss: 0.2996
Epoch 2/50:	Train Loss: 9.5465
___Validation___
 Acc: 0.8929687738418579 F1: 0.4584980309009552 Precision: 0.6236559152603149 Recall: 0.36250001192092896
Running Loss: 0.2965
Running Loss: 0.2960
Epoch 3/50:	Train Loss: 9.4550
___Validation___
 Acc: 0.888671875 F1: 0.41478440165519714 Precision: 0.6047903895378113 Recall: 0.31562501192092896
Running Loss: 0.2851
Running Loss: 0.2856
Epoch 4/50:	Train Loss: 9.1241
___Validation___
 Acc: 0.883984386920929 F1: 0.294536828994751 Precision: 0.6138613820075989 Recall: 0.19374999403953552
Running Loss: 0.2755
Running Loss: 0.2738
Epoch 5/50:	Train Loss: 8.7720
___Validation___
 Acc: 0.882031261920929 F1: 0.3198198080062866 Precision: 0.5725806355476379 Recall: 0.22187499701976776
Running Loss: 0.2637
Ru

## VGGNet-Inspired

 w/ loss reweighting or class balancing (pos_weight)


In [ ]:
train(50, FractureCNN())

Using device: cuda
tensor([  10.8557,    7.4093,   11.8227, 1204.3330,   11.0936,   10.5527,
          20.6527])
Running Loss: 0.0427
Running Loss: 0.0411
tensor([32,  0, 32,  0,  0, 25,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 27,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 27,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 23,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 22,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 21,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 28,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 25,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 26,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0, 24,  0], device='cuda:0')
Epoch 1/50:	Train Loss: 1.294
___Validation___
 Acc: 0.599 F1: 0.130 Precision: 0.075 Recall: 0.462
Running Loss: 0.0431
Running Loss: 0.0412
tensor([32,  0, 32,  0,  0,  0,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0,  0,  0], device='cuda:0')
tensor([32,  0, 32,  0,  0,  0,  0], device='cuda:0')
t